# D8 — b_U frequency prior (one model at a time)

Pre-registration: DECISIONS.md 2026-07-05 (D8, amended with prior-art disclosure).
H1 frozen: rho(b_U, freq) >= +0.3 at 12B, same sign all scales. Weights-only.

**Upload two files to this Colab session:** this notebook + `src/d8_frequency_prior.py`.

Flow: run Setup once → edit Cell 1 (model name) → run Cell 2 → repeat for all six →
run Calibration once → zip and bring it home. Cell 2 loads, runs, and **frees memory
internally**, so there is no separate cache-clear cell this time.

## Setup

In [9]:
# Cell 00: Colab Stuff
import os

try:
    import google.colab
    IN_COLAB = True
    print("Running as a Colab notebook")
    os.system('pip install transformer_lens==2.17.0 --quiet')
    os.system('pip install transformers==4.57.6 --quiet')
    os.system('pip install --upgrade numpy --quiet')
    # NOTE: legacy 'pip install --upgrade numpy' removed 2026-07-05 —
    # on current Colab images it desyncs numpy from the preinstalled
    # scipy's compiled ABI (ImportError: _center from numpy._core.umath).
    print("Dependencies installed")
except ImportError:
    IN_COLAB = False

Running as a Colab notebook
Dependencies installed


In [1]:
# Cell 0: Imports
import sys
import torch
from pathlib import Path

# Colab: files are in /content/
# Local: notebook is in notebooks/, project root is one level up
if Path('/content/data').exists():
    PROJECT_ROOT = Path('/content')
else:
    PROJECT_ROOT = Path.cwd().parent
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /


In [2]:
from transformer_lens import HookedTransformer
import transformer_lens.utils as utils

In [3]:
# Cell 0: Imports (expects d8_frequency_prior.py uploaded next to this notebook)
import sys
from pathlib import Path

assert Path('d8_frequency_prior.py').exists(), (
    'Upload src/d8_frequency_prior.py to this session first '
    '(Files sidebar, drag and drop).')
sys.path.insert(0, '.')
from d8_frequency_prior import run_d8, calibrate, MODELS
print('Ready. Models in the suite:', MODELS)

Ready. Models in the suite: ['pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-2.8b', 'pythia-6.9b', 'pythia-12b']


## One model per visit
Edit Cell 1, run Cell 2. Repeat until all six rows exist (Cell 3 shows the ledger).

In [4]:
# Cell 1: Model name variable
# pythia-160m | pythia-410m | pythia-1b | pythia-2.8b | pythia-6.9b | pythia-12b
model_name = "pythia-160m"

In [5]:
# Cell 2: Run D8 for this model
# Loads the model (TL defaults), extracts b_U + colsum, computes the frozen
# proxy correlations, saves d8_bU_<model>.csv, appends the ledger row,
# and frees GPU memory before returning. No generation, seal untouched.
run_d8('.', models=[model_name])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/375M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Loaded pretrained model pythia-160m into HookedTransformer
D8 complete -> results/d8_frequency_prior/d8_summary.md


In [7]:
# Cell 3: Ledger peek — which rows and b_U vectors exist so far
from pathlib import Path
led = Path('results/d8_frequency_prior/d8_per_scale.csv')
print(led.read_text() if led.exists() else 'no ledger yet')
print('b_U vectors on disk:',
      sorted(p.name for p in Path('results/d8_frequency_prior').glob('d8_bU_*.csv'))
      if led.exists() else '—')
print('NOTE: d8_summary.md reflects only the most recent run;the per-scale CSV above is the ledger of record.')

model,d_vocab,b_U_norm,tl_version,dtype,rho_bU_freqproxy_full,p_full,rho_bU_freqproxy_trimmed,p_trimmed,n_trimmed,rho_colsum_freqproxy_full,rho_colsum_freqproxy_trimmed,bU_vector_saved
pythia-160m,50304,5647.1025390625,2.17.0,torch.float32,0.31879519166383835,0.0,0.3216432011679974,0.0,49520,-0.003847184323257408,-0.0033497986851790808,True

b_U vectors on disk: ['d8_bU_pythia-160m.csv']
NOTE: d8_summary.md reflects only the most recent run;the per-scale CSV above is the ledger of record.


## Calibration — run ONCE, after all six b_U vectors exist
~500 Infini-gram calls at 1 qps ≈ 9 polite minutes. Crash-safe: the counts cache
checkpoints every 25 calls and the cell is resumable — rerun it and it picks up
where it left off, never re-asking a cached question. If Colab's IP still draws
403s despite pacing, download `data/infini_gram_calibration_counts.csv` (Cell 5)
and resume with `calibrate('.')` from the M5 — same cache, same command.

In [ ]:
# Cell 4: Calibrate (offline-first; joins cached Pile counts vs saved b_U vectors)
calibrate('.')

## Bring it home

In [ ]:
# Cell 5: Zip + download (loss class: ephemeral /content)
import os
os.system('zip -r d8_results.zip results/d8_frequency_prior data/infini_gram_calibration_counts.csv')
from google.colab import files
files.download('d8_results.zip')